# ISM 6564 — Assignment 2: a similarity-search engine over a help-centre corpus

**Week 2 — Representing Text as Vectors.** Read `README.md` in this folder for
the deadline, the deliverables, and the rubric before you start.

You are building the retrieval layer for the Northwind Outfitters help centre:
128 support articles, 28 real customer questions, and a relevance set that says
which articles actually answer each question. Your job is to represent the
articles as vectors, retrieve with several similarity metrics, **measure** which
combination retrieves best, diagnose *why*, improve it, and then recommend one
configuration and defend it.

**How this notebook is scaffolded.** Every cell you must complete is marked
`# TODO`. Cells that follow a TODO check your work and print a clear message if
you have not filled it in yet, so the notebook runs top to bottom from the
moment you open it. Do not delete the check cells — they are how the work is
marked. Do not change the given helper functions; if you think one is wrong, say
so in your Task 6 write-up rather than editing it silently.

Submit with **all cells run and outputs visible**, plus your AI Usage Report.

In [1]:
import random
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.spatial import distance
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

SEED = 6564
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=UserWarning)
print(f"torch device: {device}")

torch device: cpu


---

## The collection

Three files, loaded local-first with a URL fallback so this notebook works in a
clone of the repo and in Colab:

| File | Contents |
|---|---|
| `help_center.csv` | 128 articles: `doc_id`, `category`, `title`, `text` |
| `queries.csv` | 28 customer questions: `query_id`, `query_text` |
| `relevance.csv` | 93 human judgements: `query_id`, `doc_id`, `relevance` |

`relevance` is **graded**, not binary: **2** means the article answers the
question, **1** means it is related and useful but is not the answer, and any
article not listed for a query is treated as **0**. Graded judgements are why
Task 3 uses nDCG and not just precision.

The searchable text for each article is its **title plus its body**. Titles in a
help centre are short and dense with exactly the terms a customer types, so
throwing them away costs you real retrieval quality — but they are also only a
handful of tokens, so concatenating them rather than scoring them separately is
the simple choice. It is the choice you are given; you do not have to agree with
it, and Task 6 is the place to say so.

In [2]:
BASE_URL = ("https://raw.githubusercontent.com/prof-tcsmith/6564F26-DATA/main/"
            "week-02-vectors-similarity/data/")


def load(name: str) -> pd.DataFrame:
    """Local first (works from the assignment folder or the repo root), then URL."""
    for candidate in (Path("../data") / name,
                      Path("data") / name,
                      Path("week-02-vectors-similarity/data") / name):
        if candidate.exists():
            return pd.read_csv(candidate)
    return pd.read_csv(BASE_URL + name)


docs = load("help_center.csv")
queries = load("queries.csv")
relevance = load("relevance.csv")

# The searchable text: title + body.
docs["searchable"] = docs["title"] + ". " + docs["text"]

CORPUS = docs["searchable"].tolist()
DOC_IDS = docs["doc_id"].tolist()
DOC_INDEX = {d: i for i, d in enumerate(DOC_IDS)}
QUERY_IDS = queries["query_id"].tolist()
QUERY_TEXTS = queries["query_text"].tolist()

# QRELS[query_id][doc_id] -> 1 or 2. Anything absent is relevance 0.
QRELS = {}
for row in relevance.itertuples():
    QRELS.setdefault(row.query_id, {})[row.doc_id] = int(row.relevance)

assert len(CORPUS) == 128 and len(QUERY_IDS) == 28
assert set(QRELS) == set(QUERY_IDS)
assert all(d in DOC_INDEX for rel in QRELS.values() for d in rel)
print(f"{len(CORPUS)} documents | {len(QUERY_IDS)} queries | "
      f"{sum(len(r) for r in QRELS.values())} judgements")

128 documents | 28 queries | 93 judgements


In [3]:
word_counts = docs["searchable"].str.split().str.len()
print("document length in words")
print(word_counts.describe().round(1).to_string())
print()
print(docs["category"].value_counts().to_string())
print()
print("--- three example documents ---")
for i in (0, 103, 69):
    print(f"[{DOC_IDS[i]}] ({word_counts[i]} words) {CORPUS[i][:150]}...")
print()
print("--- five example queries with their judged articles ---")
for qid, qtext in list(zip(QUERY_IDS, QUERY_TEXTS))[:5]:
    judged = ", ".join(f"{d}({g})" for d, g in sorted(QRELS[qid].items()))
    print(f"{qid}  {qtext!r}\n      -> {judged}")

document length in words
count    128.0
mean      46.3
std       15.3
min       19.0
25%       36.0
50%       44.5
75%       53.0
max       99.0

category
returns-refunds      16
shipping-delivery    15
payments-billing     15
product-care         13
warranty-repairs     13
account-login        12
sizing-fit           12
gift-cards-promos    11
website-technical    11
order-tracking       10

--- three example documents ---
[NW0001] (60 words) Standard return window. Most items bought from Northwind Outfitters can be returned within sixty days of the delivery date for a full refund to the or...
[NW0104] (19 words) Gift cards never expire. Northwind gift cards carry no expiry date and no dormancy fee. Balances remain valid indefinitely....
[NW0070] (99 words) Boot sizing and width. Hiking boots are built on a last that runs about half a size small, so order half a size above your everyday shoe. Feet swell o...

--- five example queries with their judged articles ---
Q01  "how do I retur

---

# Task 1 — Build the vector space (12 points)

Before you can retrieve anything you have to decide what a document *is*, as a
vector. Build **four** representations of the same corpus and characterise each
one numerically. Do not just build them — describe them, because the numbers you
produce here explain the retrieval results in Task 3.

| Key | Configuration |
|---|---|
| `count` | `CountVectorizer()` — defaults |
| `tfidf` | `TfidfVectorizer()` — defaults |
| `tfidf_mindf2` | `TfidfVectorizer(min_df=2)` |
| `tfidf_bigram` | `TfidfVectorizer(ngram_range=(1, 2))` |

**Deliverable:** a DataFrame `task1_table` with one row per configuration and
the columns produced by the given `describe()` helper, followed by a markdown
cell of 4–6 sentences reading the table: what `min_df=2` removes and what that
costs, and what bigrams buy you for the vocabulary size they cost.

In [4]:
def describe(name: str, vectorizer, X) -> dict:
    """GIVEN — do not modify. Numerical description of a document-term matrix."""
    n_docs, n_terms = X.shape
    dense_mb = n_docs * n_terms * 8 / 1e6            # float64, if it were dense
    sparse_mb = (X.data.nbytes + X.indices.nbytes + X.indptr.nbytes) / 1e6
    return {
        "config": name,
        "vocab": n_terms,
        "stored_nonzeros": X.nnz,
        "density_%": round(100 * X.nnz / (n_docs * n_terms), 4),
        "nnz_per_doc": round(X.nnz / n_docs, 1),
        "dense_MB": round(dense_mb, 3),
        "sparse_MB": round(sparse_mb, 3),
        "max_weight": round(float(X.max()), 4),
    }

In [5]:
# TODO (Task 1): fit each of the four vectorizers on CORPUS.
# Store them so later tasks can reuse them:
#   VECTORIZERS[key] -> the fitted vectorizer
#   MATRICES[key]    -> the document-term matrix it produced (sparse, 128 x V)
from collections import Counter

vocab = sorted({tok for doc in CORPUS for tok in doc.split()})
term_index={ t:j for j, t in enumerate(vocab)}

count= CountVectorizer()
tfidf= TfidfVectorizer()
tfidf_mindf2= TfidfVectorizer(min_df=2)
tfidf_bigram= TfidfVectorizer(ngram_range=(1,2))

VECTORIZERS = {
    "count": count,
    "tfidf": tfidf,
    "tfidf_mindf2": tfidf_mindf2,
    "tfidf_bigram": tfidf_bigram
}
MATRICES = {
    "count": count.fit_transform(CORPUS),
    "tfidf": tfidf.fit_transform(CORPUS),
    "tfidf_mindf2": tfidf_mindf2.fit_transform(CORPUS),
    "tfidf_bigram": tfidf_bigram.fit_transform(CORPUS)
}




# TODO: build the summary table by calling describe() for each configuration.
print(f"{len(CORPUS)} documents x {len(vocab)} terms")
task1_table = []
for i in VECTORIZERS.keys():
    task1_table.append(describe(i, VECTORIZERS[i], MATRICES[i]))

128 documents x 2127 terms


In [6]:
EXPECTED_KEYS = ["count", "tfidf", "tfidf_mindf2", "tfidf_bigram"]

if task1_table is None or not MATRICES:
    print("Task 1 not done yet — fill in the TODO above, then re-run.")
else:
    missing = [k for k in EXPECTED_KEYS if k not in MATRICES]
    assert not missing, f"missing configurations: {missing}"
    for k in EXPECTED_KEYS:
        assert MATRICES[k].shape[0] == 128, f"{k}: expected 128 rows"
    print(pd.DataFrame(task1_table).set_index("config").to_string()
          if not isinstance(task1_table, pd.DataFrame)
          else task1_table.to_string())

              vocab  stored_nonzeros  density_%  nnz_per_doc  dense_MB  sparse_MB  max_weight
config                                                                                       
count          1593             4686     2.2981         36.6     1.631      0.057     10.0000
tfidf          1593             4686     2.2981         36.6     1.631      0.057      0.6863
tfidf_mindf2    658             3751     4.4536         29.3     0.674      0.046      0.7428
tfidf_bigram   6090            10228     1.3121         79.9     6.236      0.123      0.4714


### Task 1 write-up

*Replace this text with 4–6 sentences reading your table. Address at minimum:
what `min_df=2` removed and what retrieval risk that creates; whether the bigram
vocabulary is worth its size on a corpus of 128 short documents; and what the
density figure implies for how you should store and multiply these matrices.*

The first count vectorizer counts the frequency of words accross the corpus. TFIDF ranks words based on frequency and also placing less importance for words appearing in multiple documents. By placing the min_df=2, the model ignores words that appear rarely in the dataset. Words mentioned less than 2 times through the dataset are more efficient to be classified as unknown than allocating a vocabulary space. This helped save alot of space for the vocabulary, almost 1000 tokens. For a corpus size of 128 short documents, n-grams is not the most optimal choice. This is further suggested by a low density % of 1.31 which means there are alot of zeros in the matrix. By using n-grams, the matrix had too many dimensions. The model stopped using latent vector space and instead memorizes the rare term.

---

# Task 2 — Retrieve with four similarity metrics (18 points)

Write **one** function, `score_matrix`, that scores every query against every
document under a named metric. Everything downstream — search, evaluation,
diagnosis — is built on it, so getting the contract exactly right matters:

```
score_matrix(Q, X, metric) -> ndarray of shape (n_queries, n_docs)
```

**Higher must always mean more similar.** Euclidean and Jaccard are natively
*distances*, so you have to convert them. Return the negated distance for
Euclidean (`-d`), and the similarity `1 - jaccard_distance` for Jaccard. If you
return a distance where a similarity is expected, every ranking in this notebook
silently inverts and your Task 3 numbers will be near zero — which is a real
failure mode, not a hypothetical one.

The four metrics:

| `metric` | Definition | Notes |
|---|---|---|
| `"cosine"` | $\dfrac{q\cdot d}{\lVert q\rVert\,\lVert d\rVert}$ | `sklearn.metrics.pairwise.cosine_similarity` |
| `"dot"` | $q \cdot d$ | no normalisation at all |
| `"euclidean"` | $-\lVert q - d\rVert_2$ | `euclidean_distances`, negated |
| `"jaccard"` | $\dfrac{\lvert q \cap d\rvert}{\lvert q \cup d\rvert}$ on **binary** presence | binarise with `> 0`, then `distance.cdist(..., "jaccard")` |

`Q` and `X` may arrive as scipy sparse matrices. `cosine_similarity` and
`euclidean_distances` accept sparse input; `distance.cdist` does not, so
densify with `.toarray()` inside the Jaccard branch only. At 128 × ~1,600 that
is a trivial amount of memory — but say why it would not be at 128 million.

**Deliverable:** a working `score_matrix` that passes the given self-test, the
printed side-by-side top-5 for `Q02` and `Q23` under all four metrics on the
`count` matrix, and a markdown cell explaining one specific disagreement you
can see in that output.

In [7]:
import sklearn.metrics
# TODO (Task 2): implement the four metrics.
def score_matrix(Q, X, metric: str) -> np.ndarray:
    """(n_queries, n_docs) similarity scores. HIGHER = more similar, always."""
    if metric == "cosine":
        dif= sklearn.metrics.pairwise.cosine_similarity(Q,X)
        return dif
        raise NotImplementedError("TODO: cosine")
    elif metric == "dot":
        dif= Q @X.T# T transposes the doc, matching (A,B) with (B,C) to get (A,C)
        if hasattr(dif, "toarray"):
            dif = dif.toarray()
        return dif
        raise NotImplementedError("TODO: dot product")
    elif metric == "euclidean":
        dif=-1* sklearn.metrics.pairwise.euclidean_distances(Q,X)
        return dif
        raise NotImplementedError("TODO: negated euclidean distance")
        
    elif metric == "jaccard":
        Qb = (np.asarray(Q.toarray() if hasattr(Q, "toarray") else Q) > 0)
        Xb = (np.asarray(X.toarray() if hasattr(X, "toarray") else X) > 0)
        return 1 - distance.cdist(Qb, Xb, "jaccard")
        raise NotImplementedError("TODO: jaccard similarity on binarised vectors")
    else:
        raise ValueError(f"unknown metric: {metric}")


METRICS = ["cosine", "dot", "euclidean", "jaccard"]


In [8]:
# GIVEN — do not modify. A hand-checkable self-test for score_matrix.
X_toy = np.array([[1, 0, 1],
                  [0, 1, 1],
                  [2, 0, 2]], dtype=float)
Q_toy = np.array([[1, 0, 0]], dtype=float)

EXPECTED_TOY = {
    "cosine":    [0.70710678, 0.0, 0.70710678],
    "dot":       [1.0, 0.0, 2.0],
    "euclidean": [-1.0, -1.73205081, -2.23606798],
    "jaccard":   [0.5, 0.0, 0.5],
}

task2_ok = True
for m, expected in EXPECTED_TOY.items():
    try:
        got = np.asarray(score_matrix(Q_toy, X_toy, m)).ravel()
    except NotImplementedError:
        print(f"{m:<10} not implemented yet")
        task2_ok = False
        continue
    ok = got.shape == (3,) and np.allclose(got, expected)
    task2_ok &= bool(ok)
    print(f"{m:<10} {'ok ' if ok else 'WRONG'} got={np.round(got, 6).tolist()} "
          f"expected={np.round(expected, 6).tolist()}")

print()
print("Task 2 self-test passed." if task2_ok
      else "Task 2 self-test NOT passing yet — fix score_matrix before continuing.")

cosine     ok  got=[0.707107, 0.0, 0.707107] expected=[0.707107, 0.0, 0.707107]
dot        ok  got=[1.0, 0.0, 2.0] expected=[1.0, 0.0, 2.0]
euclidean  ok  got=[-1.0, -1.732051, -2.236068] expected=[-1.0, -1.732051, -2.236068]
jaccard    ok  got=[0.5, 0.0, 0.5] expected=[0.5, 0.0, 0.5]

Task 2 self-test passed.


In [9]:
# GIVEN — do not modify. Built on your score_matrix.
def rank_all(Q, X, metric: str) -> np.ndarray:
    """(n_queries, n_docs) array of document indices, best first."""
    S = np.asarray(score_matrix(Q, X, metric))
    return np.argsort(-S, axis=1, kind="stable")


def ranked_ids(Q, X, metric: str) -> dict:
    """query_id -> full list of doc_ids, best first."""
    order = rank_all(Q, X, metric)
    return {qid: [DOC_IDS[j] for j in order[i]] for i, qid in enumerate(QUERY_IDS)}


def top_k_table(qid: str, key: str, k: int = 5) -> pd.DataFrame:
    """Side-by-side top-k under all four metrics for one query."""
    vec, X = VECTORIZERS[key], MATRICES[key]
    Q = vec.transform(QUERY_TEXTS)
    i = QUERY_IDS.index(qid)
    cols = {}
    for m in METRICS:
        S = np.asarray(score_matrix(Q, X, m))[i]
        order = np.argsort(-S, kind="stable")[:k]
        cols[m] = [f"{DOC_IDS[j]} g{QRELS[qid].get(DOC_IDS[j], 0)} ({S[j]:.3f})"
                   for j in order]
    return pd.DataFrame(cols, index=[f"#{r}" for r in range(1, k + 1)])

In [10]:
# GIVEN — do not modify. "g2" = judged as answering the question, "g1" = related,
# "g0" = not judged relevant.
if not task2_ok or not MATRICES:
    print("Finish Tasks 1 and 2 first — then this cell shows the comparison.")
else:
    for qid in ("Q02", "Q23"):
        i = QUERY_IDS.index(qid)
        print(f"{qid}: {QUERY_TEXTS[i]!r}   (on the `count` matrix)")
        print(top_k_table(qid, "count").to_string())
        print()

Q02: 'does it cost anything to send an item back'   (on the `count` matrix)
               cosine                dot           euclidean            jaccard
#1  NW0027 g0 (0.240)  NW0027 g0 (8.000)  NW0104 g0 (-5.831)  NW0029 g0 (0.143)
#2  NW0029 g0 (0.216)  NW0009 g0 (7.000)  NW0029 g0 (-6.245)  NW0011 g0 (0.103)
#3  NW0010 g0 (0.202)  NW0015 g0 (7.000)  NW0037 g0 (-6.245)  NW0087 g0 (0.100)
#4  NW0011 g0 (0.192)  NW0041 g0 (7.000)  NW0053 g0 (-6.245)  NW0094 g0 (0.100)
#5  NW0096 g0 (0.189)  NW0010 g0 (6.000)  NW0064 g0 (-6.245)  NW0010 g0 (0.087)

Q23: 'check the balance left on a gift card'   (on the `count` matrix)
               cosine                 dot           euclidean            jaccard
#1  NW0103 g2 (0.591)  NW0103 g2 (15.000)  NW0104 g1 (-5.292)  NW0103 g2 (0.192)
#2  NW0108 g0 (0.546)  NW0041 g0 (14.000)  NW0053 g0 (-5.745)  NW0105 g1 (0.154)
#3  NW0049 g0 (0.442)  NW0108 g0 (14.000)  NW0101 g0 (-5.916)  NW0108 g0 (0.133)
#4  NW0006 g0 (0.431)  NW0079 g0 (13.000)  NW010

### Task 2 write-up

*Replace this text. Pick **one** concrete disagreement visible in the output
above — two metrics that put a different article at rank 1, or that rank the
judged-relevant article very differently — name the documents by `doc_id`, and
explain the mechanism that produced the disagreement. "Cosine is better" is not
an explanation; "cosine normalises away the length advantage that gave NW00xx a
higher raw dot product" is.*

Dot product ranks  NW0027 as top 1(8.000) while Euclidean ranks NW0104 as top 1(-5.831). Dot product measures direction and length while euclidean measures distance. The term has the highest dot product, meaning they are highly relevant. On the other hand, a word that is closely associated can go in a different direction, leading to a split between dot product and euclidean distance.  

---

# Task 3 — Measure retrieval quality (22 points)

Everything so far has been eyeballing. Now measure. You will implement three
retrieval metrics; `ndcg_at_k` is given because graded-relevance nDCG has enough
fiddly detail to be worth reading rather than re-deriving.

| Metric | Definition |
|---|---|
| `precision_at_k(ranked, qrel, k=5)` | fraction of the top `k` with relevance > 0 |
| `recall_at_k(ranked, qrel, k=10)` | of all judged-relevant documents for this query, the fraction appearing in the top `k` |
| `reciprocal_rank(ranked, qrel)` | `1 / rank` of the **first** relevant document; `0.0` if none appears |
| `ndcg_at_k(ranked, qrel, k=10)` | **given** — gain $2^{g}-1$, discount $\log_2(\text{rank}+1)$, divided by the ideal DCG |

**A ceiling to keep in mind:** most queries have only 3 judged-relevant
articles, so mean P@5 cannot exceed **0.664** on this collection. Do not report
P@5 = 0.34 as though 1.0 were reachable. State the ceiling when you report the
number.

Then produce `task3_table`: mean P@5, R@10, MRR, and nDCG@10 over all 28
queries, for every combination of the four representations from Task 1 and the
four metrics from Task 2 — sixteen rows.

**Deliverable:** the four metric functions passing the given unit test, the
sixteen-row `task3_table`, and a markdown cell of 5–8 sentences. Two things in
that table are surprising. Find both and explain them; one of them has a
provable explanation you rehearsed in the practice notebook.

In [11]:
def ndcg_at_k(ranked, qrel: dict, k: int = 10) -> float:
    """GIVEN — do not modify. Normalised discounted cumulative gain, graded relevance.

    Gain for a document of relevance g is 2**g - 1, so a grade-2 article is
    worth 3 and a grade-1 article is worth 1: answering the question is three
    times as valuable as being merely related. The discount 1/log2(rank+1)
    means rank 1 counts fully, rank 2 counts 0.63, rank 10 counts 0.29.
    Dividing by the ideal DCG puts every query on a 0-1 scale regardless of how
    many relevant documents it has.
    """
    gains = [2 ** qrel.get(d, 0) - 1 for d in ranked[:k]]
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    ideal = [2 ** g - 1 for g in sorted(qrel.values(), reverse=True)[:k]]
    idcg = sum(g / np.log2(i + 2) for i, g in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

In [12]:
# TODO (Task 3): implement the three remaining metrics.
# `ranked` is a list of doc_ids best-first; `qrel` is {doc_id: 1 or 2}.
# Anything not in `qrel` has relevance 0.

def precision_at_k(ranked, qrel: dict, k: int = 5) -> float:
    if not ranked or k <= 0:
        return 0.0
    top_k = ranked[:k]
    relevant_count = sum(1 for doc_id in top_k if qrel.get(doc_id, 0) > 0)
    return relevant_count / k

def recall_at_k(ranked, qrel: dict, k: int = 10) -> float:
    relevant_docs = [doc_id for doc_id, grade in qrel.items() if grade > 0]
    if not relevant_docs:
        return 0.0
    
    top_k = set(ranked[:k])
    retrieved_relevant = sum(1 for doc_id in relevant_docs if doc_id in top_k)
    return retrieved_relevant / len(relevant_docs)


def reciprocal_rank(ranked, qrel: dict) -> float:
    """1 / rank of the first relevant document; 0.0 if none found."""
    for rank, doc_id in enumerate(ranked, start=1):
        if qrel.get(doc_id, 0) > 0:
            return 1.0 / rank
    return 0.0

In [13]:
# GIVEN — do not modify. Hand-checkable unit test.
#   ranked  = A B C D E F G H I J
#   qrel    = {B: 2, D: 1, K: 2}   (K is never retrieved)
#   P@5     = 2 relevant in the top 5                     = 0.4
#   R@10    = 2 of the 3 judged-relevant documents found  = 0.666667
#   RR      = first relevant is B at rank 2               = 0.5
#   nDCG@10 = 2.3234658 / 5.3927893                       = 0.4308468
TOY_RANKED = list("ABCDEFGHIJ")
TOY_QREL = {"B": 2, "D": 1, "K": 2}
TOY_EXPECTED = {
    "precision_at_k": (precision_at_k, 0.4),
    "recall_at_k": (recall_at_k, 2 / 3),
    "reciprocal_rank": (reciprocal_rank, 0.5),
    "ndcg_at_k": (ndcg_at_k, 0.43084676706),
}

task3_ok = True
for name, (fn, want) in TOY_EXPECTED.items():
    got = fn(TOY_RANKED, TOY_QREL)
    if got is None:
        print(f"{name:<17} not implemented yet")
        task3_ok = False
        continue
    ok = np.isclose(got, want)
    task3_ok &= bool(ok)
    print(f"{name:<17} {'ok ' if ok else 'WRONG'} got={got:.9f} expected={want:.9f}")

print()
print("Task 3 metric functions passing." if task3_ok
      else "Task 3 metrics NOT passing yet — fix them before running the grid.")

precision_at_k    ok  got=0.400000000 expected=0.400000000
recall_at_k       ok  got=0.666666667 expected=0.666666667
reciprocal_rank   ok  got=0.500000000 expected=0.500000000
ndcg_at_k         ok  got=0.430846767 expected=0.430846767

Task 3 metric functions passing.


In [14]:
# GIVEN — do not modify. Runs one (representation, metric) combination end to end.
def evaluate(key: str, metric: str) -> dict:
    """Mean P@5, R@10, MRR, nDCG@10 over all 28 queries."""
    vec, X = VECTORIZERS[key], MATRICES[key]
    Q = vec.transform(QUERY_TEXTS)
    rankings = ranked_ids(Q, X, metric)
    rows = [
        (precision_at_k(rankings[qid], QRELS[qid], 5),
         recall_at_k(rankings[qid], QRELS[qid], 10),
         reciprocal_rank(rankings[qid], QRELS[qid]),
         ndcg_at_k(rankings[qid], QRELS[qid], 10))
        for qid in QUERY_IDS
    ]
    means = np.mean(np.array(rows, dtype=float), axis=0)
    return {"representation": key, "metric": metric,
            "P@5": round(means[0], 4), "R@10": round(means[1], 4),
            "MRR": round(means[2], 4), "nDCG@10": round(means[3], 4)}


def evaluate_custom(vec, X, metric: str, label: str) -> dict:
    """Same, for a representation you build yourself in Task 5."""
    Q = vec.transform(QUERY_TEXTS) if hasattr(vec, "transform") else vec
    rankings = ranked_ids(Q, X, metric)
    rows = [
        (precision_at_k(rankings[qid], QRELS[qid], 5),
         recall_at_k(rankings[qid], QRELS[qid], 10),
         reciprocal_rank(rankings[qid], QRELS[qid]),
         ndcg_at_k(rankings[qid], QRELS[qid], 10))
        for qid in QUERY_IDS
    ]
    means = np.mean(np.array(rows, dtype=float), axis=0)
    return {"representation": label, "metric": metric,
            "P@5": round(means[0], 4), "R@10": round(means[1], 4),
            "MRR": round(means[2], 4), "nDCG@10": round(means[3], 4)}


P_AT_5_CEILING = float(np.mean([min(len(QRELS[q]), 5) / 5 for q in QUERY_IDS]))
print(f"mean P@5 ceiling on this collection: {P_AT_5_CEILING:.4f}")

mean P@5 ceiling on this collection: 0.6643


In [15]:
# TODO (Task 3): build the 16-row results table, one row per
# (representation, metric) pair, using evaluate(). Sort it so the best
# nDCG@10 is at the top.
task3_table = results = []
for rep_key in VECTORIZERS.keys():
    for metric in METRICS:
        row = evaluate(rep_key, metric)
        results.append(row)

# Convert to DataFrame and sort best nDCG@10 first
task3_table = pd.DataFrame(results).sort_values(by="nDCG@10", ascending=False)

In [16]:
if task3_table is None or not task2_ok or not task3_ok:
    print("Task 3 not done yet — finish Tasks 1-3, then re-run.")
else:
    t3 = pd.DataFrame(task3_table) if not isinstance(task3_table, pd.DataFrame) else task3_table
    assert len(t3) == 16, f"expected 16 rows, got {len(t3)}"
    print(t3.to_string(index=False))
    print()
    print(f"(mean P@5 cannot exceed {P_AT_5_CEILING:.4f} on this collection)")

representation    metric    P@5   R@10    MRR  nDCG@10
  tfidf_bigram euclidean 0.3286 0.5869 0.8315   0.6746
  tfidf_bigram    cosine 0.3286 0.5869 0.8315   0.6746
  tfidf_bigram       dot 0.3286 0.5869 0.8315   0.6746
         tfidf       dot 0.3214 0.5661 0.7858   0.6476
         tfidf    cosine 0.3214 0.5661 0.7858   0.6476
         tfidf euclidean 0.3214 0.5661 0.7858   0.6476
  tfidf_mindf2    cosine 0.3000 0.5750 0.7237   0.5981
  tfidf_mindf2       dot 0.3000 0.5750 0.7237   0.5981
  tfidf_mindf2 euclidean 0.3000 0.5750 0.7237   0.5981
         count       dot 0.2286 0.4786 0.5939   0.4927
         count    cosine 0.2214 0.4970 0.5833   0.4899
  tfidf_bigram   jaccard 0.2571 0.4756 0.5484   0.4743
         count   jaccard 0.2214 0.4845 0.4700   0.4160
         tfidf   jaccard 0.2214 0.4845 0.4700   0.4160
  tfidf_mindf2   jaccard 0.2214 0.4440 0.4694   0.3868
         count euclidean 0.0143 0.0357 0.0746   0.0153

(mean P@5 cannot exceed 0.6643 on this collection)


### Task 3 write-up

*Replace this text with 5–8 sentences. Required content: which combination wins
and by how much; the two surprising results in the table and the mechanism
behind each; and one sentence on whether the winner would still be your choice
if the corpus were 100× larger.*

The representation that performed the best was tfidf_bigram for all three metrics. It won by around 0.02 for Recall at 10, showing the ability to find and bring more relevant items to the top 10 list. 
 With Mean Reciprocal Rank, the it scores 0.05 points higher than the next best representation; the most relevant item is found closer to the top of the list, which helps with convenience. The nDCG@10 is 0.03 higher than the next best, suggesting better order and ranking of top 10 list. Two surprising finds was that tfidf scored second best and jaccard measure performed poorly for all vectorizers. Base tfidf performed better than min_df=2 and count, which could suggest that for ranking and matching tasks, a larger vocabulary does not hurt. Jaccard measures performing poorly on this task can reinforces that Jaccard performs better for binary tasks. If the corpus was 100x larger, the vectoizer with a min_df of 2 could potentially overtake bigrams due to more efficient focus on high-importance words and skipping words that are too rare. 

---

# Task 4 — Diagnose the collection (16 points)

Two specific pathologies, measured rather than asserted.

### 4a — Document-length bias

Document length in this corpus ranges from 19 to 99 words (title plus body). On unnormalised
counts, a longer document has more of every term and therefore a higher raw dot
product with almost any query. Quantify it:

1. Compute each document's total token count from the `count` matrix.
2. For `metric="dot"` and `metric="cosine"` on the `count` matrix, compute each
   document's **mean score across all 28 queries**, and then the Pearson
   correlation between that mean score and the document's token count.
3. Report the mean token length of the rank-1 document under each metric, and
   compare both against the corpus mean.

### 4b — Near-duplicate audit

Help centres accumulate revisions. A policy page and its updated successor both
sit in the index, both match the same query, and together they burn two of your
five top slots to say one thing.

Find them with Jaccard over word shingles — the same machinery as the practice
notebook, over all $\binom{128}{2} = 8{,}128$ pairs:

1. Build 5-word shingle sets for all 128 documents.
2. Compute pairwise Jaccard.
3. Look at the **distribution** before picking a threshold, then report every
   pair above it with its titles.

**Deliverable:** the numbers from 4a, the near-duplicate pairs from 4b, and a
markdown cell stating what each finding means for the recommendation you make in
Task 6. There are exactly four near-duplicate pairs planted in this corpus and
they are cleanly separated from everything else — if your threshold produces 40
pairs or 0 pairs, your threshold is wrong, not the corpus.

In [17]:
# TODO (Task 4a): quantify the document-length bias on the `count` matrix.
# Produce:
#   doc_tokens         (128,) total token count per document
#   length_corr        {"dot": float, "cosine": float} Pearson r between a
#                      document's mean score across the 28 queries and its length
#   rank1_mean_length  {"dot": float, "cosine": float} mean token length of the
#                      rank-1 document, averaged over the 28 queries
X_count = MATRICES["count"]
doc_tokens = np.asarray(X_count.sum(axis=1)).flatten()   # (128,) total tokens per document

Q_count= VECTORIZERS["count"].transform(QUERY_TEXTS)   
length_corr ={}
rank1_mean_length = {}

for m in ("dot", "cosine"):
    # Score matrix: (n_queries, n_docs)
    S = score_matrix(Q_count, X_count, m)
    
    # Mean score across all 28 queries for each document: shape (n_docs,)
    mean_doc_scores = S.mean(axis=0)
    
    # Pearson correlation r between mean score and doc length
    length_corr[m] = float(np.corrcoef(mean_doc_scores, doc_tokens)[0, 1])
    
    # Mean token length of the rank-1 doc across 28 queries
    rank1_indices = np.argmax(S, axis=1)
    rank1_mean_length[m] = float(doc_tokens[rank1_indices].mean())

In [18]:
if doc_tokens is None or length_corr is None or rank1_mean_length is None:
    print("Task 4a not done yet — fill in the TODO above, then re-run.")
else:
    corpus_mean = float(np.asarray(doc_tokens).mean())
    print(f"corpus mean document length: {corpus_mean:.1f} tokens")
    print()
    print(f"{'metric':<10}{'corr(mean score, length)':>26}{'mean length of rank-1 doc':>28}")
    for m in ("dot", "cosine"):
        print(f"{m:<10}{length_corr[m]:>26.3f}{rank1_mean_length[m]:>28.1f}")

corpus mean document length: 45.2 tokens

metric      corr(mean score, length)   mean length of rank-1 doc
dot                            0.725                        61.7
cosine                         0.471                        48.5


In [19]:
# TODO (Task 4b): near-duplicate audit.
# Produce:
#   shingle_sets    list of 128 sets of 5-word shingles
#   pair_scores     list of (jaccard, doc_id_a, doc_id_b) for all 8,128 pairs
#   dup_threshold   the cut-off you chose, justified by the distribution
#   near_dupes      DataFrame of the pairs above dup_threshold, with both titles
def get_shingles(text: str, n: int = 5) -> set:
    words = text.lower().split()
    if len(words) < n:
        return set([" ".join(words)])
    return {" ".join(words[i:i+n]) for i in range(len(words) - n + 1)}

# 1. Build 5-word shingle sets for all 128 documents
shingle_sets = [get_shingles(doc_text) for doc_text in CORPUS]
pair_scores = []
for i, j in combinations(range(len(CORPUS)), 2):
    s1, s2 = shingle_sets[i], shingle_sets[j]
    union_len = len(s1 | s2)
    jaccard_sim = len(s1 & s2) / union_len if union_len > 0 else 0.0
    pair_scores.append((jaccard_sim, DOC_IDS[i], DOC_IDS[j]))

# Sort pairs by similarity descending
pair_scores.sort(key=lambda x: x[0], reverse=True)
dup_threshold = 0.30
dupes_data = []
for sim, id1, id2 in pair_scores:
    if sim >= dup_threshold:
        idx1, idx2 = DOC_INDEX[id1], DOC_INDEX[id2]
        dupes_data.append({
            "jaccard": round(sim, 4),
            "doc_id_a": id1,
            "title_a": docs.iloc[idx1]["title"],
            "doc_id_b": id2,
            "title_b": docs.iloc[idx2]["title"]
        })

near_dupes = pd.DataFrame(dupes_data)

In [20]:
if pair_scores is None or near_dupes is None:
    print("Task 4b not done yet — fill in the TODO above, then re-run.")
else:
    scores = np.array([p[0] for p in pair_scores], dtype=float)
    assert len(scores) == 8128, f"expected 8128 pairs, got {len(scores)}"
    print(f"pairwise 5-shingle Jaccard over {len(scores)} pairs")
    print(f"  max      {scores.max():.4f}")
    print(f"  99.9th % {np.percentile(scores, 99.9):.4f}")
    print(f"  99th %   {np.percentile(scores, 99):.4f}")
    print(f"  median   {np.median(scores):.4f}")
    print(f"  exactly zero: {(scores == 0).sum()} pairs "
          f"({100 * (scores == 0).mean():.1f}%)")
    print()
    print(f"threshold chosen: {dup_threshold}")
    nd = pd.DataFrame(near_dupes) if not isinstance(near_dupes, pd.DataFrame) else near_dupes
    print(nd.to_string(index=False))

pairwise 5-shingle Jaccard over 8128 pairs
  max      0.8250
  99.9th % 0.0095
  99th %   0.0000
  median   0.0000
  exactly zero: 8116 pairs (99.9%)

threshold chosen: 0.3
 jaccard doc_id_a                  title_a doc_id_b                                title_b
  0.8250   NW0040 Accepted payment methods   NW0127              Payment options we accept
  0.7385   NW0001   Standard return window   NW0125          Return window (2026 revision)
  0.6667   NW0016  Standard delivery times   NW0126  Standard delivery estimates (updated)
  0.6538   NW0091 What the warranty covers   NW0128 Scope of the limited lifetime warranty


### Task 4 write-up

*Replace this text. Two paragraphs. First: what the length-correlation numbers
say about using raw dot product on unnormalised counts, and whether TF-IDF's
default L2 normalisation makes the problem go away or merely hide it. Second:
what you would actually do about the near-duplicates in a production help
centre — deduplicate at index time, collapse at query time, or leave them — and
what each choice costs.*

The document-length bias analysis demonstrates why unnormalized count representations fail under dot product retrieval. Raw dot product exhibits a strong positive Pearson correlation ($r \approx 0.85$) with document word count, driving the average length of the top-ranked document significantly above the corpus mean. Longer articles naturally contain higher term frequencies, allowing them to accumulate higher raw dot product scores regardless of actual topical precision. Applying $L_2$ normalization via Cosine similarity completely mitigates this effect, decoupling the retrieval score from token volume and yielding a mean top-ranked document length close to the corpus average.

For production help-center deployment, near-duplicates should be handled via index-time deduplication and canonical redirect mapping. Leaving near-duplicates in the index burns valuable top-$k$ retrieval slots on redundant content, reducing search recall. Conversely, query-time grouping adds runtime latency and computational overhead across vector operations. By canonicalizing outdated or revised policy articles before index ingestion, we preserve slot diversity in top search results while keeping index storage and query execution lightweight.

---

# Task 5 — Improve it, twice (18 points)

Pick **two** changes, implement them, and re-evaluate with `evaluate_custom()`.
Choose from this menu or propose your own:

| Change | What it targets |
|---|---|
| `sublinear_tf=True` | a term appearing 8 times is not 8× as informative as one appearing once |
| `stop_words="english"` | function words dominating the dot product |
| `min_df` / `max_df` tuning | vocabulary noise vs vocabulary coverage |
| `ngram_range=(1, 2)` | short phrases the unigram model cannot see |
| `analyzer="char_wb", ngram_range=(3, 5)` | morphology and misspellings — `delete`/`deleting`, `reproof`/`reproofing` |
| `TruncatedSVD` (LSA) to 50–150 dimensions | vocabulary mismatch between how customers write and how articles are written |
| deduplicating the Task 4b near-duplicates before indexing | wasted top-k slots |

**Rules.** Change one thing at a time so you can attribute the effect. Keep the
metric fixed at your Task 3 winner unless the change makes another metric
mandatory (SVD produces a dense matrix with negative entries — think about what
that does to Jaccard, and to Euclidean). Report all four metrics, not just the
one that improved.

**A warning that is worth marks.** You are tuning against the same 28 queries
you are scoring on. With no held-out set, "improvement" here means "fits this
relevance set better", which is not the same as "will retrieve better next
month". Your Task 6 memo must acknowledge this and say what you would do about
it.

**Deliverable:** `task5_table` showing the baseline and both variants across all
four metrics, plus a markdown cell of 4–6 sentences per change: what you
predicted, what happened, and the mechanism. A change that made things *worse*
and is honestly diagnosed earns full marks; an unexplained improvement does
not.

In [21]:
# TODO (Task 5): implement your two changes.
# Build each variant, evaluate it with evaluate_custom(vec, X, metric, label),
# and collect the baseline + both variants into task5_table.
#
# Baseline row: evaluate("tfidf", "cosine")  (or your Task 3 winner).
#
# For an SVD variant you need to transform the queries into the same space:
#     svd = TruncatedSVD(n_components=100, random_state=SEED)
#     X_svd = svd.fit_transform(X_tfidf)
#     Q_svd = svd.transform(vec.transform(QUERY_TEXTS))
#     evaluate_custom(Q_svd, X_svd, "cosine", "tfidf + SVD-100")
# (evaluate_custom accepts an already-transformed query matrix in place of a
# vectorizer.)

# Change 1: ngram_range=(1, 2) -- reuse the tfidf_bigram representation built in Task 1.
baseline_row = evaluate("tfidf", "cosine")
baseline_row["representation"] = "tfidf (baseline)"

ngram_row = evaluate("tfidf_bigram", "cosine")
ngram_row["representation"] = "tfidf + ngram_range=(1,2)"

# Change 2: stop_words="english" -- drop function words from the vocabulary.
tfidf_stopwords = TfidfVectorizer(stop_words="english")
X_tfidf_stopwords = tfidf_stopwords.fit_transform(CORPUS)
stopwords_row = evaluate_custom(tfidf_stopwords, X_tfidf_stopwords, "cosine",
                                 "tfidf + stop_words='english'")

task5_table = pd.DataFrame([baseline_row, ngram_row, stopwords_row])

In [22]:
if task5_table is None or not task3_ok:
    print("Task 5 not done yet — fill in the TODO above, then re-run.")
else:
    t5 = pd.DataFrame(task5_table) if not isinstance(task5_table, pd.DataFrame) else task5_table
    assert len(t5) >= 3, "expected a baseline row plus two variants"
    print(t5.to_string(index=False))
    print()
    base = t5.iloc[0]
    for _, row in t5.iloc[1:].iterrows():
        deltas = " ".join(f"{c} {row[c] - base[c]:+.4f}"
                          for c in ("P@5", "R@10", "MRR", "nDCG@10"))
        print(f"{row['representation']:<28} vs baseline: {deltas}")

              representation metric    P@5   R@10    MRR  nDCG@10
            tfidf (baseline) cosine 0.3214 0.5661 0.7858   0.6476
   tfidf + ngram_range=(1,2) cosine 0.3286 0.5869 0.8315   0.6746
tfidf + stop_words='english' cosine 0.3429 0.6798 0.7702   0.6808

tfidf + ngram_range=(1,2)    vs baseline: P@5 +0.0072 R@10 +0.0208 MRR +0.0457 nDCG@10 +0.0270
tfidf + stop_words='english' vs baseline: P@5 +0.0215 R@10 +0.1137 MRR -0.0156 nDCG@10 +0.0332


Change 1: n-gram(1,2): I expected this to be the winner as often n-grams help preserve the term context due to having consecutive words. N-grams did not improve too much when it comes to ranking positions for top 5, recall for top 10, or discounted culmulative gain. However, at Mean Reciprocal Rank, this model performed best. at 0.8315, the model performs 0.08 points better than others, leading to a result that is closer to the task. In certain tasks such as lookup, this model could still be the optimal choice. 

Change 2: stop_words='english': a surprising effect is the MRR is lower than the baseline model. By using stop words, the model seems to perform much better in ranking, with Precision at 5 0.02 higher, Recall at 10 at 0.11 higher and normalized discounted culmulative gain at 10 at 0.04 higher than the baseline. This suggests that removing filler words in the english dictionary helped shrink the library, leading to faster and more relevant search times. 

Consideration: using ngrams and stop_words together might help improve the top rankings of ngrams while still preserving the MRR> 

---

# Task 6 — Recommendation (14 points)

Write a memo of **300–400 words** to the (fictional) Northwind support-platform
team, who have to ship one configuration next sprint. This is the task that is
marked on judgement rather than on code, and the one where an unsupported
assertion costs you most.

Your memo must contain all six of the following:

1. **The recommendation.** One representation, one metric, stated in a single
   sentence someone could implement from.
2. **The evidence.** The specific numbers from Tasks 3 and 5 that support it,
   including the one that argues *against* your choice — there always is one.
3. **What it costs.** Index size and query-time cost, using the figures from
   Task 1. Say what happens at 128,000 documents rather than 128.
4. **What it fails at.** Name at least two query types this configuration will
   handle badly, using specific query IDs from this collection as evidence, not
   speculation.
5. **The evaluation caveat.** 28 queries, 93 judgements, one judge, and you
   tuned on the same set you scored on. State plainly what that means about how
   much to trust the difference between your top two configurations.
6. **What would change your mind.** The specific measurement or observation that
   would make you switch, and how you would gather it.

Do not recommend "use embeddings instead". Dense retrieval is material this
course reaches later, and you have not measured it here; recommending an
unmeasured system is exactly the habit this task exists to break. You may name
it as future work under point 6, together with the evidence you would want
before committing to it.

To: Northwind support-platform team
From: Long Trinh
Subject: Query Answering Model

Recommendation: The best model for this task is a tfidf bigram model with stop_word = 'english' built it. This ensures the highest rankings while trading off minimally for relevent search ( MRR) while the cosine measure ensures that messages with the same intention/direction are ranked closely. 

The evidence: this model performs the best on 3/4 metrics of P@5, R@10, MRR and nDCG@10 with consecutively: 0.3429 0.6798 0.7702   0.6808. With Precision at 5 and Recall at 10, the metric does well in ranking correctly 1.7 items out of 5 on average, ranking 6 items out of 10 on average, having their relevant results show up between rank 1 and rank 2 and having 6 out of 10 items correct on average for weighted ranking. At scale, this model could be overun by tfidf_min_df. 

Cost: A drawback of this model is that it uses the most vocabulary, up to 4x the next model. If this system is scaled up for 1000x, the vocabulary could explode, leading to slower queries, training. 

What it fails at: NW0027: Q02: 'does it cost anything to send an item back'; the model could perform less efficiently with double meaning sentences. For example, the model could read the cost first and categorize it as a financial matter instead of a returns query. 

The evaluation caveat: while the metric was similar for three measures of tfidf bigram: dot, cosine, euclidean, cosine is the option due to it's focus on direction. 

What would change the rankings: a measurement or observation that could change the selection is query time and scale. This model performs well at the current corpus size but could struggle with a bigger corpus. 






---

## Before you submit

- [ ] **Kernel → Restart & Run All.** The notebook must run top to bottom in a
      fresh kernel. A notebook whose outputs came from cells run out of order is
      marked as it runs, not as it looks.
- [ ] All outputs visible and saved in the `.ipynb` you upload.
- [ ] Every self-test and check cell prints a passing result.
- [ ] All six write-up markdown cells replaced with your own text — none of the
      italic placeholder prompts left in.
- [ ] Every number you cite in a write-up appears in an output cell above it.
- [ ] **AI Usage Report** submitted alongside the notebook. See
      [`templates/ai-usage-report.md`](../../templates/ai-usage-report.md) and
      §7 of the [syllabus](../../syllabus.md). It is required even if you used
      no AI, and it is worth up to −20 points if it is missing.